## Topic 7: Metadata Filtering in Practice

### Concept, Intuition, Why It Exists

- Semantic search alone answers "which chunks are most similar in meaning to this query?" — but in a real system, the answer to a query often depends not just on similarity but on *where* the answer should come from. "What is the premature withdrawal penalty?" is a very different question when asked in the context of a customer FAQ than when asked while reviewing an internal SOP, even though the query text is identical.
- **Metadata filtering** is the mechanism that adds a structured constraint on top of semantic search: "find me the most semantically similar chunks, but only among chunks from this specific source, document type, or version." It combines vector search (what does the query mean?) with structured filtering (where should the answer come from?).
- Without filtering, a single Qdrant collection holding FAQs, SOPs, policy documents, and customer records would return a blend of all of them for any query — the top-k results could include chunks from three different document types, none of them the one the user actually needed. Metadata filtering is what turns a search that returns "anything relevant" into a search that returns "the most relevant thing from the right place."
- In this project's context: filtering by `doc_type` scopes search to only FAQ documents, only policy documents, or only product guides. Filtering by `source` scopes it to a specific file. Both are useful for different situations — covered in the code below.

### Internal Working

- Qdrant evaluates the filter **during** HNSW traversal, not after. The traversal only visits and considers nodes whose payload matches the filter conditions. This means the k results returned already satisfy the filter — you always get exactly k matching results back, not k results from which you then discard the non-matching ones.
- Qdrant's filter model is built around conditions combined with logical operators:
- `must` — all conditions must be true (AND logic)
- `should` — at least one condition must be true (OR logic)
- `must_not` — the condition must be false (NOT logic)
- These can be nested and combined to express complex scoping rules like "only FAQ or policy documents, but not from the draft source."
- A `FieldCondition` with `MatchValue` is the simplest filter — "this field must equal exactly this value." Other match types include `MatchAny` (field value is one of a list), range filters for numeric fields, and nested object conditions for deeply structured payloads.

### How It's Implemented In This Project

- Every point in the collection has `doc_type` and `source` in its payload, populated at upsert time from the Document's metadata dict — the same metadata that the ingestion chapter's loaders already attached. No extra work is needed at ingest time to enable filtering; the metadata is already there.
- A `doc_type` payload index was added in Topic 5 so filtered searches on that field are fast at scale — without the index, filtering requires scanning all payload records, which defeats the purpose at large corpus sizes.
- Filtering is used in two patterns in this project: scoping by document type (e.g. "only search FAQs") for query routing, and scoping by source file (e.g. "only search the current policy document, not the draft") for version-aware retrieval.

### Real-World Issues, Edge Cases, Debugging

- **Over-filtering returns too few results**: if a filter restricts search to a very small subset of points (e.g. filtering to a single source file that has only 3 chunks) and k=5 is requested, Qdrant can only return 3. The returned list being shorter than k is not an error — it's correct behavior, but it can surprise code that assumes `len(results) == k` is always true.
- **Wrong payload key name silently returns full results or no results**: if the filter references `"document_type"` but the payload stores it as `"doc_type"`, Qdrant doesn't raise an error — it simply treats the missing key as not matching the condition, returning either zero results or falling back to a full unfiltered scan depending on configuration. This is the same payload-schema-consistency problem named in Topic 5, showing up here as a silent retrieval failure.
- **Filtering and HNSW recall interact**: a very aggressive filter on a large collection can cause HNSW traversal to miss valid matching points if they're not reachable via the graph from the entry point within the `ef` search budget. Qdrant handles this gracefully but it's worth knowing that very restrictive filters on large collections can return fewer results than expected even when more matching points exist.
- **No filter is not the same as an empty filter**: passing `query_filter=None` searches the entire collection. Passing an empty `Filter(must=[])` also searches the entire collection. But constructing a filter with a condition that matches nothing (e.g. `MatchValue(value="nonexistent_type")`) returns zero results. All three look similar in code but behave differently in edge cases worth testing explicitly.

### Design Decisions & Trade-offs

- Filtering at the Qdrant level vs. filtering in Python after retrieval: Qdrant-level filtering is always the right choice when the filter is known before the query — it's faster (traversal skips non-matching nodes), cheaper (fewer results fetched and discarded), and returns exactly k useful results rather than k results of which some fraction are useful. Python-level post-filtering is only appropriate when the filter condition can't be expressed in Qdrant's filter model (e.g. a filter that depends on runtime computation over the retrieved text itself).
- One collection with payload filtering vs. multiple collections per document type: one collection with filtering is simpler to maintain, query, and update. Multiple collections only make sense when document types need completely different vector sizes (different embedding models), or when access control requires physical separation that payload filtering alone can't enforce.
- Adding `doc_type` and `source` to every point's payload even for sources where filtering might never be needed: the cost is negligible (a few extra bytes per point) and the benefit is that filtering is always available if a requirement changes. Not adding them and then needing to add them later means a full re-ingest.

### Alternatives & When To Use Each

- Qdrant payload filtering during search — the default choice whenever the filter is known before the query, the filter field is already in the payload, and a payload index exists on that field.
- Python-level post-filtering — only when the filter logic requires computation that can't be expressed as a Qdrant condition, accepting the cost of over-fetching and discarding.
- Multiple collections — only when different document types need different vector sizes or require complete physical access separation at the infrastructure level.
- Hybrid: filter by `doc_type` in Qdrant, then re-rank by recency or other business logic in Python — a common production pattern where Qdrant handles the semantic + structural scoping and Python handles business rules that are too complex for a simple payload condition.

### Common Mistakes & Production Failures

- Referencing a payload field name that doesn't match what was stored at upsert time, and getting silent retrieval failures (zero results or wrong results) rather than an error.
- Not adding a payload index on `doc_type` or `source` before going to production, and discovering that filtered queries are doing full payload scans at scale.
- Assuming `len(results) == k` after a filtered search and having downstream code crash when a very restrictive filter returns fewer than k results.
- Using post-search Python filtering as the default approach because it was simpler to implement initially, then discovering it consistently returns fewer than k useful results on queries where most chunks are the wrong type.

### Lead-Level Interview Questions

**Q: A user asks "what is the FD penalty for premature withdrawal?" and the system should only search FAQ documents. How do you implement this and why is filtering in Qdrant better than filtering in Python after retrieval?**
A: Add a `doc_type` payload condition to the `query_points` call restricting to `"faq"`. Qdrant applies this during HNSW traversal so only FAQ chunks are even considered — the k results returned are all FAQs. Python-level post-filtering would mean fetching more than k results and discarding non-FAQ ones, which either wastes the overfetch or returns fewer than k useful results if most chunks happen to be non-FAQ. Qdrant-level filtering is faster, cheaper, and guarantees k useful results.

**Q: A filter on `doc_type` is returning zero results even though you can see matching points were upserted. What are the three most likely causes?**
A: First, a payload key name mismatch — the filter says `"doc_type"` but the payload stored it as `"document_type"` or similar. Second, a value case mismatch — the filter says `"FAQ"` but the payload stores `"faq"` (MatchValue is case-sensitive). Third, no payload index on `doc_type` combined with a very large collection, causing the filter to time out or fail silently. Check the actual stored payload on a specific point using `client.retrieve()` to confirm exactly what field names and values were written at upsert time.

**Q: How would you implement "search only the most recent version of each document type, not older versions"?**
A: Store a `version` field in the payload at upsert time, populated from the document's version metadata. At query time, filter on both `doc_type` and `version` — either by explicitly specifying the current version value (if you track it centrally) or by tagging the current version with an `is_current=True` boolean payload field at ingest time and filtering on that. The second approach is more robust because you don't need to know the current version number at query time, only that a point is or isn't current.

### Hidden Concepts Worth Knowing

- **Payload index type matters for filter performance**: a keyword index (string equality) is different from an integer range index. Filtering on `doc_type` (a string field) needs a keyword index. Filtering on `page` (an integer field) needs an integer index. Using the wrong index type or no index type degrades filtered search performance at large scale in ways that are invisible at small scale.
- **Qdrant's `should` (OR) logic and minimum match count**: `should` conditions can have a `minimum_should_match` parameter specifying how many of the OR conditions must be satisfied — more expressive than a plain OR, useful for "must match at least 2 of these 5 criteria" style filtering that would otherwise require complex nested logic.

### Revision Summary

> Metadata filtering adds a structured constraint on top of semantic search — "find me the most similar chunks, but only from this document type or source." Qdrant applies filters during HNSW traversal, not after, so the k results returned already satisfy the filter. The three filter operators are `must` (AND), `should` (OR), and `must_not` (NOT). Common failure modes: payload key name mismatch returns zero results silently; no payload index means full payload scan at scale; over-aggressive filtering on a small subset returns fewer than k results without an error.

In [1]:
"""
qdrant_filtering.py
---------------------
Metadata filtering in practice -- scoping search by doc_type and source.

Shows:
  - must (AND) filtering: only return chunks matching one doc_type
  - should (OR) filtering: return chunks from any of several doc_types
  - must_not (NOT) filtering: exclude a specific source from results
  - combined filtering: must + must_not together
  - the over-filtering edge case: filter leaves fewer points than k
  - how to debug a silent filter failure using client.retrieve()

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    PayloadSchemaType,
    Filter,
    FieldCondition,
    MatchValue,
    MatchAny,     # "field value must be one of this list" -- OR across values
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_filtering_demo"
VECTOR_SIZE = 384
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

client = QdrantClient(":memory:")
model = SentenceTransformer(MODEL_NAME)

# corpus with a mix of doc_types and sources -- filtering is only useful when there's variety
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy_v2.json", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to death of the depositor.",
     "source": "FD_Policy_v2.json", "doc_type": "policy"},
    {"text": "DRAFT: Premature withdrawal penalty under review, may change to 0.5 percent.",
     "source": "FD_Policy_draft.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the penalty for early FD closure? A 1 percent penalty applies.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "Can I break my FD before maturity? Yes, with a penalty on the interest rate.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "Step 1: Verify customer identity. Step 2: Check FD status before processing.",
     "source": "FD_SOP.json", "doc_type": "sop"},
]


def make_point_id(source: str, index: int) -> int:
    # deterministic ID so re-upserts update in place
    return int(hashlib.md5(f"{source}::{index}".encode()).hexdigest()[:8], 16)


def setup() -> None:
    # create collection and add payload index on doc_type
    existing = [c.name for c in client.get_collections().collections]
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )

    # payload index on doc_type -- makes filtered searches fast at scale
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="doc_type",
        field_schema=PayloadSchemaType.KEYWORD,
    )

    # also index source -- needed for source-level filtering later
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="source",
        field_schema=PayloadSchemaType.KEYWORD,
    )

    # embed and upsert all chunks
    texts = [c["text"] for c in CHUNKS]
    embeddings = model.encode(texts, normalize_embeddings=True)
    points = [
        PointStruct(
            id=make_point_id(CHUNKS[i]["source"], i),
            vector=embeddings[i].tolist(),
            payload={
                "text": CHUNKS[i]["text"],
                "source": CHUNKS[i]["source"],
                "doc_type": CHUNKS[i]["doc_type"],
            },
        )
        for i in range(len(CHUNKS))
    ]
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"Setup complete: {client.get_collection(COLLECTION_NAME).points_count} points\n")


def search(query: str, k: int = 3, query_filter=None, label: str = "") -> None:
    """Generic search helper -- reused by all demo functions below."""
    query_vector = model.encode(query, normalize_embeddings=True).tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,  # None = no filter = search everything
        limit=k,
        with_payload=True,
    ).points

    print(f"--- {label} (k={k}, got={len(results)}) ---")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] [{r.payload['source']}]")
        print(f"           {r.payload['text'][:70]}")
    print()


QUERY = "What is the penalty for closing an FD early?"


def demo_no_filter() -> None:
    """No filter -- searches all 8 chunks across all doc types."""
    search(QUERY, k=4, query_filter=None, label="No filter (all doc types)")


def demo_must_single() -> None:
    """must with one condition: only return FAQ chunks.
    The k results are all FAQs -- no post-search discarding needed."""
    search(
        QUERY, k=3,
        query_filter=Filter(
            must=[
                # must = ALL conditions in this list must be true (AND logic)
                FieldCondition(key="doc_type", match=MatchValue(value="faq"))
            ]
        ),
        label="Filter: doc_type must be 'faq'",
    )


def demo_should_or() -> None:
    """should with two conditions: return FAQ OR policy chunks.
    At least one of the should conditions must match."""
    search(
        QUERY, k=4,
        query_filter=Filter(
            should=[
                # should = at least one of these must be true (OR logic)
                FieldCondition(key="doc_type", match=MatchValue(value="faq")),
                FieldCondition(key="doc_type", match=MatchValue(value="policy")),
            ]
        ),
        label="Filter: doc_type should be 'faq' OR 'policy'",
    )


def demo_match_any() -> None:
    """MatchAny is a cleaner way to express the same OR across values."""
    search(
        QUERY, k=4,
        query_filter=Filter(
            must=[
                # MatchAny = field value must be one of these -- equivalent to the should above
                FieldCondition(key="doc_type", match=MatchAny(any=["faq", "policy"]))
            ]
        ),
        label="Filter: doc_type in ['faq', 'policy'] using MatchAny",
    )


def demo_must_not() -> None:
    """must_not: exclude the draft source entirely.
    Useful for version-aware retrieval -- only search approved documents."""
    search(
        QUERY, k=4,
        query_filter=Filter(
            must_not=[
                # must_not = this condition must be FALSE (NOT logic)
                FieldCondition(key="source", match=MatchValue(value="FD_Policy_draft.json"))
            ]
        ),
        label="Filter: exclude draft source (must_not)",
    )


def demo_combined() -> None:
    """Combined must + must_not: only policy docs, but not the draft version.
    This is the "only search current approved policy" pattern."""
    search(
        QUERY, k=3,
        query_filter=Filter(
            must=[
                # only policy documents
                FieldCondition(key="doc_type", match=MatchValue(value="policy"))
            ],
            must_not=[
                # but not the draft
                FieldCondition(key="source", match=MatchValue(value="FD_Policy_draft.json"))
            ],
        ),
        label="Filter: policy only, exclude draft (must + must_not)",
    )


def demo_over_filtering() -> None:
    """Over-filtering edge case: SOP doc_type has only 1 chunk in the collection.
    Requesting k=3 returns only 1 result -- not an error, just correct behavior.
    Code that assumes len(results) == k will break here."""
    search(
        QUERY, k=3,
        query_filter=Filter(
            must=[FieldCondition(key="doc_type", match=MatchValue(value="sop"))]
        ),
        label="Over-filtering: only 1 SOP chunk exists but k=3 requested",
    )


def demo_debug_silent_failure() -> None:
    """How to debug a silent filter failure -- wrong key name returns zero results.
    Use client.retrieve() to see exactly what's stored before assuming the filter is right."""

    print("--- Debug: wrong payload key name (silent failure) ---")
    # this filter uses 'document_type' but the payload stores 'doc_type' -- returns nothing
    query_vector = model.encode(QUERY, normalize_embeddings=True).tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        # wrong key name -- will silently return zero results, no error raised
        query_filter=Filter(
            must=[FieldCondition(key="document_type", match=MatchValue(value="faq"))]
        ),
        limit=3,
        with_payload=True,
    ).points
    print(f"  Results with wrong key 'document_type': {len(results)} (expected 0)")

    # fix: use client.retrieve() to inspect what's actually stored in the payload
    all_ids = [make_point_id(CHUNKS[i]["source"], i) for i in range(len(CHUNKS))]
    sample = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=[all_ids[0]],  # look at the first point
        with_payload=True,
        with_vectors=False,
    )
    print(f"  Actual payload keys stored: {list(sample[0].payload.keys())}")
    print(f"  Use these exact key names in your filter conditions\n")


# ======================================================================
# Run all demos in order
# ======================================================================

setup()
demo_no_filter()
demo_must_single()
demo_should_or()
demo_match_any()
demo_must_not()
demo_combined()
demo_over_filtering()
demo_debug_silent_failure()

C:\Users\pauls\AppData\Local\Temp\ipykernel_52452\2066581024.py:77: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


Setup complete: 8 points

--- No filter (all doc types) (k=4, got=4) ---
  score=0.8909 | [faq] [FD_FAQ.json]
           What is the penalty for early FD closure? A 1 percent penalty applies.
  score=0.6248 | [faq] [FD_FAQ.json]
           Can I break my FD before maturity? Yes, with a penalty on the interest
  score=0.5689 | [policy] [FD_Policy_v2.json]
           Premature withdrawal incurs a 1 percent penalty on the applicable rate
  score=0.5678 | [policy] [FD_Policy_v2.json]
           This penalty does not apply if the FD is closed due to death of the de

--- Filter: doc_type must be 'faq' (k=3, got=2) ---
  score=0.8909 | [faq] [FD_FAQ.json]
           What is the penalty for early FD closure? A 1 percent penalty applies.
  score=0.6248 | [faq] [FD_FAQ.json]
           Can I break my FD before maturity? Yes, with a penalty on the interest

--- Filter: doc_type should be 'faq' OR 'policy' (k=4, got=4) ---
  score=0.8909 | [faq] [FD_FAQ.json]
           What is the penalty for ear